# 04 — Evaluation: coherence vs baseline LDA

**Professor feedback:** *Augmenting LDA with semantic embeddings should significantly improve the coherence of the identified topics in large scientific corpora. Calculating a Coherence Score to quantitatively compare your hybrid results against a baseline LDA model.*

This notebook reads `results/metrics.csv` produced by `python run_pipeline.py` and reports C_v / NPMI / UMass, silhouette, diversity, and the α sweep.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import RESULTS_DIR, FIGURES_DIR

metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
metrics

In [ ]:
best = (
    metrics.sort_values("c_v", ascending=False)
    .groupby(["corpus", "model"], as_index=False)
    .first()[["corpus", "model", "n_topics", "c_v", "c_npmi", "u_mass", "diversity", "silhouette"]]
)
best

In [ ]:
print("Relative C_v lift of Hybrid vs LDA (best K per model)\n")
for corpus, g in best.groupby("corpus"):
    lda = g.loc[g["model"] == "LDA", "c_v"]
    hyb = g.loc[g["model"] == "Hybrid-BERT-LDA", "c_v"]
    if len(lda) and len(hyb) and float(lda.iloc[0]) != 0:
        lift = 100 * (float(hyb.iloc[0]) - float(lda.iloc[0])) / abs(float(lda.iloc[0]))
        print(f"{corpus:16s}  LDA={float(lda.iloc[0]):.3f}  Hybrid={float(hyb.iloc[0]):.3f}  lift={lift:+.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (corpus, g) in zip(axes, metrics.groupby("corpus")):
    for model, gg in g.groupby("model"):
        ax.plot(gg["n_topics"], gg["c_v"], marker="o", label=model)
    ax.set_title(corpus)
    ax.set_xlabel("K")
    ax.set_ylabel("C_v")
    ax.legend()
fig.suptitle("Coherence C_v vs baseline LDA")
plt.show()

In [ ]:
sil = pd.read_csv(RESULTS_DIR / "silhouette.csv")
alpha = pd.read_csv(RESULTS_DIR / "alpha_sweep.csv")
display(sil.pivot_table(index=["corpus", "model"], columns="reducer", values="silhouette"))
display(alpha)

Figures written by the pipeline live in `results/figures/`. Open the Streamlit app for interactive topic exploration, temporal trends, and related-work search:

`streamlit run app/streamlit_app.py`